# Convo AI: Phase 1 (text, Chinese, ordering at a cafe)
An AI waiter (Qwen 2.5 3B Instruct) chats with an A2-level Chinese learner and corrects their mistakes.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.
The last cell is a live chat for the demo video; it waits for you to type (type `q` to stop).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again"
MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to("cuda")
print("Loaded", MODEL)

## 1. The waiter character
The prompt tells the model to stay in character and answer in exactly two lines: a reply, and a correction (or 无 if there's no mistake).

In [ ]:
SYSTEM = """你是一家咖啡店的服务员。顾客是一位刚移民到中国的中文学习者（A2水平）。
用简单的中文回答，像真正的服务员一样，不要跳出角色。
每次只用下面的格式回答，只有两行：
回复：<你作为服务员的回答，用简单的中文>
纠正：<如果顾客的句子有错误，写出正确的句子，然后用英文简单解释为什么。如果没有错误，只写「无」>"""

def ask(history):
    msgs = [{"role": "system", "content": SYSTEM}] + history
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse(text):
    reply, fix = "", ""
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("回复"): reply = line[3:].strip()
        if line.startswith("纠正"): fix = line[3:].strip()
    return reply, fix

print(ask([{"role": "user", "content": "我要一个咖啡。"}]))

## 2. Our test set
The quiz lives in `test_sentences.csv` in the GitHub repo, so the whole team edits one file. If `learner == correct`, the sentence has no mistake.
After editing the CSV on GitHub, wait ~5 minutes before rerunning (GitHub caches the file).

**TODO:** first draft written with Claude's help. A Chinese speaker on the team must check every row, fix anything wrong, and add your own sentences. Say so in the README.

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/jackthescammer/Convo-ai-chatbot/main/test_sentences.csv")
df["has_mistake"] = df.learner != df.correct
tune = df.sample(10, random_state=0)  # use these to improve the prompt
test = df.drop(tune.index)            # never look at these until the final score
print(len(df), "sentences |", df.has_mistake.sum(), "with mistakes |", len(test), "held out for testing")

## 3. Tune on the 10 practice sentences
Look at the output. If the waiter breaks character or the format is wrong, change `SYSTEM` above and rerun. Only use these 10 sentences for this.

In [ ]:
import re
def bare(s): return re.sub(r"[\s，。？！,.?!「」]", "", s)

def run(data):
    out = data.copy()
    out["raw"] = [ask([{"role": "user", "content": s}]) for s in out.learner]
    out["reply"], out["fix"] = zip(*[parse(r) for r in out.raw])
    out["flagged"] = [f != "" and not f.startswith("无") for f in out.fix]
    out["fix_matches"] = [bare(g) in bare(f) for g, f in zip(out.correct, out.fix)]
    out["in_character"] = [bool(re.search(r"[一-鿿]", r)) for r in out.reply]
    return out

tune_res = run(tune)
tune_res[["learner", "correct", "reply", "fix", "flagged"]]

## 4. Final score on held-out sentences (baseline vs Convo)
Don't change the prompt after running this, or the test isn't fair.
Baseline: a waiter that always says 听起来不错！ ("sounds good!") and never corrects anything.

In [ ]:
res = run(test)
m, ok = res[res.has_mistake], res[~res.has_mistake]
print(f"BASELINE: caught 0/{len(m)} mistakes, 0/{len(ok)} false alarms")
print(f"CONVO:    caught {m.flagged.sum()}/{len(m)} mistakes")
print(f"CONVO:    correction matches our answer exactly: {m.fix_matches.sum()}/{len(m)}")
print(f"CONVO:    false alarms on correct sentences: {ok.flagged.sum()}/{len(ok)}")
print(f"CONVO:    stayed in character (Chinese reply in the right format): {res.in_character.sum()}/{len(res)}")
res.to_csv("results.csv", index=False)

## 5. Where it fails
Exact matching is strict: a correction worded differently counts as a miss here. So also download `results.csv` (Files panel on the left) and have a Chinese speaker mark each correction right or wrong. Report that number too.

In [ ]:
fails = res[(res.has_mistake & ~res.fix_matches) | (~res.has_mistake & res.flagged)]
for _, r in fails.head(5).iterrows():
    print("LEARNER:", r.learner, "\nCORRECT:", r.correct, "\nCONVO:  ", r.fix, "\n")

## 6. Live demo chat (for the video)
Type as the customer. Type `q` to stop.

In [ ]:
history = []
while True:
    msg = input("你: ")
    if msg.strip() in ("", "q"): break
    history.append({"role": "user", "content": msg})
    ans = ask(history)
    history.append({"role": "assistant", "content": ans})
    print(ans, "\n")